In [32]:
import sys
#from podio import root_io
import uproot 
import awkward as ak
import math
import vector
import numpy as np
path = "/work/eic/users/aabhishe/EIC_3He_5x41_XRot_pi_bc.hepmc3.tree_sim_output_recon.root"

In [ ]:


# 1. Open your EIC output file
reader = root_io.Reader(path)

# Counters
total_events = 0
events_with_truth_match = 0
events_with_proton = 0
total_protons_found = 0

# 2. Loop over events
for ievt, event in enumerate(reader.get("events")):
    total_events += 1
    rec_charged_particles = event.get("ReconstructedChargedParticles")
    hits = event.get("ForwardRomanPotHits")
    if not rec_charged_particles:
        continue

    event_has_truth_match = False
    
    # Track unique protons seen in this specific event to avoid
    # counting multiple detector layers for the same particle
    found_proton_ids_in_event = set()
    

    for hit in hits:
        # Filter out low-energy secondary background (delta-rays, soft photons)
        if hit.getMomentum().z < 20.0:
            continue

        mc_particle = hit.getParticle()

        # Check if truth particle information is linked
        if mc_particle.isAvailable():
            event_has_truth_match = True
            
            # Check for proton (PDG = 2212)
            if mc_particle.getPDG() == 2212:
                # Use the Podio/EDM object ID or address to distinguish unique particles
                proton_id = mc_particle.getObjectID()
                found_proton_ids_in_event.add(proton_id)

    if event_has_truth_match:
        events_with_truth_match += 1

    if found_proton_ids_in_event:
        events_with_proton += 1
        total_protons_found += len(found_proton_ids_in_event)

# 3. Print Summary Results
print("=" * 45)
print("             ANALYSIS SUMMARY")
print("=" * 45)
print(f"Total events analyzed:                 {total_events}")
print(f"Events with truth-matched hits:        {events_with_truth_match}")
print(f"Events with at least one proton:       {events_with_proton}")
print(f"Total unique spectator protons found:  {total_protons_found}")
if total_events > 0:
    print(f"Proton Roman Pot Acceptance/Hit Rate:  {100.0 * events_with_proton / total_events:.2f}%")
print("=" * 45)

[[4, 4, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], ..., [4, 4, 2, 1, 1, 1, 0, 0]]


In [3]:
reader= root_io.Reader(path)

tree = reader.get("events")

electrons = []

for ievt, event in enumerate(tree):
    rec_charged_particles = event.get("ReconstructedChargedParticles")
    if not rec_charged_particles:
        continue
    if rec_charged_particles.size() > 0:
        for particle in rec_charged_particles:
            if particle.getPDG() == 11:  # PDG code for electron
                electrons.append(particle)
                

    

In [46]:
tree = uproot.open(f"{path}:events")

# Read arrays
e_prime_pdg_arrays = tree["ReconstructedChargedParticles.PDG"].array()
e_beam_pdg_arrays = tree["MCParticles.PDG"].array()
mc_gen_status = tree["MCParticles.generatorStatus"].array()

e_beam_energy = 5.0
p_beam_energy = 41.0

e_prime_mask = (e_prime_pdg_arrays == 11)

e_beam_mask = (e_beam_pdg_arrays == 11) & (mc_gen_status == 4)
p_beam_mask = (e_beam_pdg_arrays == 1000020030 ) & (mc_gen_status == 4)
# Apply masks for scattered (reco) electrons
e_prime_pz = tree["ReconstructedChargedParticles.momentum.z"].array()[e_prime_mask]
e_prime_px = tree["ReconstructedChargedParticles.momentum.x"].array()[e_prime_mask]
e_prime_py = tree["ReconstructedChargedParticles.momentum.y"].array()[e_prime_mask]
e_prime_E = tree["ReconstructedChargedParticles.energy"].array()[e_prime_mask]

# Apply masks for beam (MC truth) electrons
e_beam_pz = tree["MCParticles.momentum.z"].array()[e_beam_mask]
e_beam_px = tree["MCParticles.momentum.x"].array()[e_beam_mask]
e_beam_py = tree["MCParticles.momentum.y"].array()[e_beam_mask]
e_beam_E = ak.full_like(e_beam_pz, e_beam_energy)

p_beam_pz = tree["MCParticles.momentum.z"].array()[p_beam_mask]
p_beam_px = tree["MCParticles.momentum.x"].array()[p_beam_mask]
p_beam_py = tree["MCParticles.momentum.y"].array()[p_beam_mask]
p_beam_E = ak.full_like(p_beam_pz, p_beam_energy)



e_beam = vector.zip({
    "px": e_beam_px,
    "py": e_beam_py,
    "pz": e_beam_pz,
    "E": e_beam_E
})

e_prime = vector.zip({
    "px": e_prime_px,
    "py": e_prime_py,
    "pz": e_prime_pz,
    "E": e_prime_E
})

p_beam = vector.zip({
    "px": p_beam_px,
    "py": p_beam_py,
    "pz": p_beam_pz,
    "E": p_beam_E
})
sorted_indices = ak.argsort(e_prime.E, ascending=False)
e_prime_sorted = e_prime[sorted_indices]

e_beam_single = ak.firsts(e_beam)
p_beam_single = ak.firsts(p_beam)
e_prime_leading = ak.firsts(e_prime_sorted)

#four momentum transfer
q = e_beam_single - e_prime_leading
Q2 = -q.mass2

#bjorken x
x = Q2 / (2 * p_beam.dot(q))

print(f"Q2: {Q2[:10]}")
print(f"x: {x[:10]}")
Q2_clean = ak.to_numpy(ak.drop_none(Q2)).astype(np.float64)
x_clean = ak.to_numpy(ak.drop_none(x)).astype(np.float64)


Q2: [2.48, 2.27, 1.66, 1.43, 2.55, 1.05, 1.18, 1.52, 3.92, 20.6]
x: [[0.0133], [0.011], [0.0111], [0.0125], ..., [0.00785], [0.00532], [0.0416]]


In [47]:
import ROOT
Q2_bin_edges = np.logspace(1, 3, 101)  # 100 bins from 10^-2 to 10^2

Q2_hist = ROOT.TH1F("his1", "Q2 Distribution; Q2 [GeV^2]; Events", len(Q2_bin_edges) - 1, Q2_bin_edges)
xbj_bin_edges = np.logspace(-4, 0, 101)  # 100 bins from 10^-4 to 10^-1

x_hist = ROOT.TH1F("his2", "Bjorken x Distribution; x; Events", len(xbj_bin_edges) - 1, xbj_bin_edges)

Q2_hist.FillN(len(Q2_clean), Q2_clean, np.ones(len(Q2_clean), dtype=np.float64))
x_hist.FillN(len(x_clean), x_clean, np.ones(len(x_clean), dtype=np.float64))
canvas= ROOT.TCanvas("canvas", "Q2 and x Distributions", 800, 600)
canvas.Divide(2, 1)
canvas.cd(1)
ROOT.gPad.SetLogy()
ROOT.gPad.SetLogx()
Q2_hist.Draw()
canvas.cd(2)
ROOT.gPad.SetLogx()
ROOT.gPad.SetLogy()
x_hist.Draw()

canvas.SaveAs("/home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_distributions.pdf")


Warning in <TROOT::Append>: Replacing existing TH1: his1 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: his2 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: pdf file /home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_distributions.pdf has been created


TypeError: zip() got an unexpected keyword argument 'with_name'

In [17]:
e_prime = vector.zip({
    "px": momentum_x_arrays,
    "py": momentum_y_arrays,
    "pz": momentum_z_arrays,
    "E": energy_arrays
})

In [21]:
print(e_prime.pt[3])
print(e_prime.eta[3])

[1.15]
[-2.09]
